In [1]:
import joblib
import os
import pandas as pd

## Load Pre-trained Models

Load the Logistic Regression models and vectorizers trained in the individual notebooks.

In [2]:
models_dir = '../models'

# Load spam detection model and vectorizer
spam_model = joblib.load(os.path.join(models_dir, 'lr_model.joblib'))
spam_vectorizer = joblib.load(os.path.join(models_dir, 'lr_vectorizer.joblib'))

# Load phishing detection model and vectorizer
phishing_model = joblib.load(os.path.join(models_dir, 'phishing_lr_model.joblib'))
phishing_vectorizer = joblib.load(os.path.join(models_dir, 'phishing_lr_vectorizer.joblib'))

print("Spam detection model loaded")
print("Phishing detection model loaded")

Spam detection model loaded
Phishing detection model loaded


## Define Pipeline Classification Function

In [3]:
def classify_email(text, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer):
    """
    Classify an email through the spam → phishing pipeline.
    """
    result = {
        'text': text,
        'spam_prediction': None,
        'spam_confidence': None,
        'spam_probabilities': None,
        'phishing_prediction': None,
        'phishing_confidence': None,
        'phishing_probabilities': None,
        'final_label': None
    }
    
    # Stage 1: Spam Detection
    text_vec_spam = spam_vectorizer.transform([text])
    spam_pred = spam_model.predict(text_vec_spam)[0]
    spam_probs = spam_model.predict_proba(text_vec_spam)[0]
    
    result['spam_prediction'] = spam_pred
    result['spam_confidence'] = spam_probs[spam_pred]
    result['spam_probabilities'] = {'ham': spam_probs[0], 'spam': spam_probs[1]}
    
    # If ham (not spam), we're done
    if spam_pred == 0:
        result['final_label'] = 'HAM'
        return result
    
    # Stage 2: Phishing Detection (only for spam emails)
    text_vec_phishing = phishing_vectorizer.transform([text])
    phishing_pred = phishing_model.predict(text_vec_phishing)[0]
    phishing_probs = phishing_model.predict_proba(text_vec_phishing)[0]
    
    result['phishing_prediction'] = phishing_pred
    result['phishing_confidence'] = phishing_probs[phishing_pred]
    result['phishing_probabilities'] = {'safe': phishing_probs[0], 'phishing': phishing_probs[1]}
    
    # Final classification
    if phishing_pred == 1:
        result['final_label'] = 'PHISHING'
    else:
        result['final_label'] = 'SPAM'
    
    return result

## Pretty Print Results Function

In [4]:
def display_result(result, example_num=None):
    """
    Display classification results in a user-friendly format.
    """
    label_styles = {
        'HAM': {'icon': '✅'},
        'SPAM': {'icon': '⚠️'},
        'PHISHING': {'icon': '🚨'}
    }
    
    style = label_styles[result['final_label']]
    
    header = f"Example {example_num}" if example_num else "Classification Result"
    print(f"\n{'='*70}")
    print(f"{header}")
    print(f"{'='*70}")
    
    text = result['text']
    display_text = text[:150] + "..." if len(text) > 150 else text
    print(f"\n📧 Email: {display_text}")
    
    print(f"\n--- Stage 1: Spam Detection ---")
    spam_label = "SPAM" if result['spam_prediction'] == 1 else "HAM"
    print(f"Prediction: {spam_label}")
    print(f"Confidence: {result['spam_confidence']:.2%}")
    print(f"Probabilities: Ham={result['spam_probabilities']['ham']:.2%}, Spam={result['spam_probabilities']['spam']:.2%}")
    
    if result['phishing_prediction'] is not None:
        print(f"\n--- Stage 2: Phishing Detection ---")
        phishing_label = "PHISHING" if result['phishing_prediction'] == 1 else "SAFE (Regular Spam)"
        print(f"Prediction: {phishing_label}")
        print(f"Confidence: {result['phishing_confidence']:.2%}")
        print(f"Probabilities: Safe={result['phishing_probabilities']['safe']:.2%}, Phishing={result['phishing_probabilities']['phishing']:.2%}")
    else:
        print(f"\n--- Stage 2: Phishing Detection ---")
        print("Skipped (email is not spam)")
    
    print(f"\n{'='*70}")
    print(f"{style['icon']} FINAL CLASSIFICATION: {result['final_label']} {style['icon']}")
    print(f"{'='*70}")

## Test the Pipeline

Testing with the same examples as the Naive Bayes pipeline for comparison.

In [5]:
# Test examples covering all three categories
test_emails = [
    # Legitimate emails (expected: HAM)
    {
        'text': "Hi Team, Please find attached the quarterly report. Let me know if you have any questions. Best regards, John",
        'expected': 'HAM',
        'description': 'Legitimate business email'
    },
    {
        'text': "Meeting reminder: Project sync tomorrow at 2pm in Conference Room B. Agenda has been shared.",
        'expected': 'HAM',
        'description': 'Meeting reminder'
    },
    {
        'text': "Can you review the contract and send me your feedback by Friday? Thanks!",
        'expected': 'HAM',
        'description': 'Work request'
    },
    
    # Regular spam emails (expected: SPAM)
    {
        'text': "AMAZING DEALS! Buy one get one FREE! Limited time offer on all products! Shop now and save big!",
        'expected': 'SPAM',
        'description': 'Marketing spam'
    },
    {
        'text': "You have been selected for a special discount! Act now to receive 50% off your next purchase!",
        'expected': 'SPAM',
        'description': 'Promotional spam'
    },
    
    # Phishing emails (expected: PHISHING)
    {
        'text': "URGENT: Your account has been compromised! Click here immediately to verify your identity and secure your account before it's too late.",
        'expected': 'PHISHING',
        'description': 'Account compromise phishing'
    },
    {
        'text': "Your PayPal account has been limited. Please update your information immediately to avoid suspension. Click here to verify your account.",
        'expected': 'PHISHING',
        'description': 'PayPal phishing'
    },
    {
        'text': "Dear Customer, We detected unusual activity on your bank account. Please confirm your identity by providing your SSN and password immediately.",
        'expected': 'PHISHING',
        'description': 'Banking phishing'
    },
    {
        'text': "Congratulations! You have won $1,000,000 in the lottery! Click the link below and enter your credit card information to claim your prize now!",
        'expected': 'PHISHING',
        'description': 'Lottery scam phishing'
    }
]

# Run classification for each test email
results = []
for i, email in enumerate(test_emails, 1):
    result = classify_email(
        email['text'],
        spam_model, spam_vectorizer,
        phishing_model, phishing_vectorizer
    )
    result['expected'] = email['expected']
    result['description'] = email['description']
    results.append(result)
    display_result(result, example_num=i)


Example 1

📧 Email: Hi Team, Please find attached the quarterly report. Let me know if you have any questions. Best regards, John

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 98.40%
Probabilities: Ham=98.40%, Spam=1.60%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 2

📧 Email: Meeting reminder: Project sync tomorrow at 2pm in Conference Room B. Agenda has been shared.

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 95.26%
Probabilities: Ham=95.26%, Spam=4.74%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 3

📧 Email: Can you review the contract and send me your feedback by Friday? Thanks!

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 98.14%
Probabilities: Ham=98.14%, Spam=1.86%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 4

📧 Email: AMAZING DEALS! Buy one get one FRE

## Pipeline Accuracy Summary

In [6]:
# Create summary table
summary_data = []
for i, r in enumerate(results, 1):
    match = '✅' if r['final_label'] == r['expected'] else '❌'
    summary_data.append({
        'Example': i,
        'Description': r['description'],
        'Expected': r['expected'],
        'Predicted': r['final_label'],
        'Match': match
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("PIPELINE CLASSIFICATION SUMMARY (Logistic Regression)")
print("="*70)
print(summary_df.to_string(index=False))

# Calculate accuracy
correct = sum(1 for r in results if r['final_label'] == r['expected'])
total = len(results)
accuracy = correct / total

print(f"\n{'='*70}")
print(f"Pipeline Accuracy on Test Examples: {correct}/{total} ({accuracy:.1%})")
print(f"{'='*70}")


PIPELINE CLASSIFICATION SUMMARY (Logistic Regression)
 Example                 Description Expected Predicted Match
       1   Legitimate business email      HAM       HAM     ✅
       2            Meeting reminder      HAM       HAM     ✅
       3                Work request      HAM       HAM     ✅
       4              Marketing spam     SPAM  PHISHING     ❌
       5            Promotional spam     SPAM  PHISHING     ❌
       6 Account compromise phishing PHISHING  PHISHING     ✅
       7             PayPal phishing PHISHING  PHISHING     ✅
       8            Banking phishing PHISHING  PHISHING     ✅
       9       Lottery scam phishing PHISHING  PHISHING     ✅

Pipeline Accuracy on Test Examples: 7/9 (77.8%)


## Compare Phishing Probabilities

Let's look more closely at the phishing probabilities to see if LR can distinguish spam from phishing.

In [7]:
# Focus on emails that made it past spam detection
print("\nPhishing Detection Analysis (for emails classified as spam):")
print("="*80)

for i, r in enumerate(results, 1):
    if r['spam_prediction'] == 1:  # Only spam emails
        phishing_prob = r['phishing_probabilities']['phishing']
        safe_prob = r['phishing_probabilities']['safe']
        
        print(f"\nExample {i}: {r['description']}")
        print(f"  Expected: {r['expected']}")
        print(f"  Phishing Probability: {phishing_prob:.2%}")
        print(f"  Safe Probability: {safe_prob:.2%}")
        print(f"  Predicted: {r['final_label']} {'✅' if r['final_label'] == r['expected'] else '❌'}")


Phishing Detection Analysis (for emails classified as spam):

Example 4: Marketing spam
  Expected: SPAM
  Phishing Probability: 92.56%
  Safe Probability: 7.44%
  Predicted: PHISHING ❌

Example 5: Promotional spam
  Expected: SPAM
  Phishing Probability: 80.48%
  Safe Probability: 19.52%
  Predicted: PHISHING ❌

Example 6: Account compromise phishing
  Expected: PHISHING
  Phishing Probability: 99.87%
  Safe Probability: 0.13%
  Predicted: PHISHING ✅

Example 7: PayPal phishing
  Expected: PHISHING
  Phishing Probability: 99.48%
  Safe Probability: 0.52%
  Predicted: PHISHING ✅

Example 8: Banking phishing
  Expected: PHISHING
  Phishing Probability: 96.83%
  Safe Probability: 3.17%
  Predicted: PHISHING ✅

Example 9: Lottery scam phishing
  Expected: PHISHING
  Phishing Probability: 95.18%
  Safe Probability: 4.82%
  Predicted: PHISHING ✅


## Interactive Email Classifier

In [8]:
# Enter your own email text to classify
user_email = """
Dear valued customer,

We have detected suspicious activity on your Amazon account. 
Your account will be suspended unless you verify your information within 24 hours.

Click here to verify: http://amaz0n-security.fake-domain.com/verify

Please provide your login credentials and payment information to restore access.

Regards,
Amazon Security Team
"""

result = classify_email(
    user_email.strip(),
    spam_model, spam_vectorizer,
    phishing_model, phishing_vectorizer
)

display_result(result)


Classification Result

📧 Email: Dear valued customer,

We have detected suspicious activity on your Amazon account. 
Your account will be suspended unless you verify your information...

--- Stage 1: Spam Detection ---
Prediction: SPAM
Confidence: 91.42%
Probabilities: Ham=8.58%, Spam=91.42%

--- Stage 2: Phishing Detection ---
Prediction: PHISHING
Confidence: 98.95%
Probabilities: Safe=1.05%, Phishing=98.95%

🚨 FINAL CLASSIFICATION: PHISHING 🚨
